# LoRA Stage 1 — bootstrap a dataset of goth_d (InstantID)

Generates ~20 **varied but identity-consistent** photos of the goth_d hero from
her single reference image, so Stage 2 has enough material to train a character
LoRA. Identity is locked with **InstantID** (one face -> many images); the
prompt varies expression / lighting / background / framing.

Outputs `goth_d_dataset.zip` (images + caption .txt files, trigger word
`g0thd`). Runs on a Colab **T4**.

> InstantID has a fiddly setup (insightface, antelopev2, model downloads). If a
> cell fails, fix just that cell and re-run — the heavy downloads are cached.

In [ ]:
# 1. GPU check.
!nvidia-smi

In [ ]:
# 2. Install. Pin diffusers to a version the InstantID pipeline expects, and
#    pin pillow<12 so nothing breaks diffusers. (~3-4 min)
!pip install -q "diffusers==0.27.2" "huggingface_hub<0.26" transformers accelerate \
    safetensors insightface==0.7.3 onnxruntime opencv-python-headless "pillow<12"
print("\n*** If pip changed Pillow: Runtime > Restart session, then run from cell 3.")

In [ ]:
# 3. Guard.
import PIL
assert int(PIL.__version__.split('.')[0]) < 12, (
    'Pillow >= 12 -> Runtime > Restart session, then run from here.')
print('Pillow', PIL.__version__, 'OK')

In [ ]:
# 4. Download InstantID assets, the antelopev2 face models, and the pipeline.
from huggingface_hub import hf_hub_download, snapshot_download

# InstantID IdentityNet (ControlNet) + image-prompt adapter.
hf_hub_download("InstantX/InstantID", "ControlNetModel/config.json", local_dir="checkpoints")
hf_hub_download("InstantX/InstantID", "ControlNetModel/diffusion_pytorch_model.safetensors", local_dir="checkpoints")
hf_hub_download("InstantX/InstantID", "ip-adapter.bin", local_dir="checkpoints")

# antelopev2 face-analysis models (insightface's own URL is often dead).
snapshot_download("DIAMONIK7777/antelopev2", local_dir="models/antelopev2")

# InstantID diffusers pipeline + draw_kps helper (from the GitHub repo).
!wget -q https://raw.githubusercontent.com/InstantID/InstantID/main/pipeline_stable_diffusion_xl_instantid.py
import os
assert os.path.exists("pipeline_stable_diffusion_xl_instantid.py"), "pipeline file download failed"
print("InstantID assets ready")

In [ ]:
# 5. Read her face from the GitHub hero image -> embedding + keypoints.
import cv2
import numpy as np
from insightface.app import FaceAnalysis
from diffusers.utils import load_image
from pipeline_stable_diffusion_xl_instantid import draw_kps

app = FaceAnalysis(name="antelopev2", root=".",
                   providers=["CPUExecutionProvider"])
app.prepare(ctx_id=0, det_size=(640, 640))

HERO_URL = ("https://raw.githubusercontent.com/ParaskaBohdan/haRdQualizer/"
            "main/assets/characters/goth_d/body.png")
face_image = load_image(HERO_URL).convert("RGB")
face_cv = cv2.cvtColor(np.array(face_image), cv2.COLOR_RGB2BGR)
faces = app.get(face_cv)
assert faces, "No face detected in the hero image"
face = sorted(faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))[-1]
face_emb = face.normed_embedding
face_kps = draw_kps(face_image, face.kps)
print("face found, embedding dim:", face_emb.shape)

In [ ]:
# 6. Load the InstantID pipeline on the GPU (RealVisXL base, same as the hero).
import torch
from diffusers import ControlNetModel
from pipeline_stable_diffusion_xl_instantid import StableDiffusionXLInstantIDPipeline

controlnet = ControlNetModel.from_pretrained(
    "checkpoints/ControlNetModel", torch_dtype=torch.float16)
pipe = StableDiffusionXLInstantIDPipeline.from_pretrained(
    "SG161222/RealVisXL_V5.0", controlnet=controlnet, torch_dtype=torch.float16)
pipe.load_ip_adapter_instantid("checkpoints/ip-adapter.bin")
pipe = pipe.to("cuda")
pipe.enable_vae_tiling()
print("InstantID pipeline ready")

In [ ]:
# 7. Generate the varied dataset (identity fixed, prompt varies).
from pathlib import Path

TRIGGER = "g0thd"
BASE = ("{trigger} woman, photo of a goth woman, pale skin, dark grey eyes, "
        "dark plum lips, long black hair, blunt bangs, black gothic dress, "
        "choker, {var}, photorealistic, detailed skin")
NEG = ("nude, naked, nsfw, deformed, bad hands, extra fingers, blurry, lowres, "
       "watermark, text, cartoon, anime, 3d render, doll, child")

VARIATIONS = [
    "front view, neutral expression", "three quarter view, soft smile",
    "looking to the side, serious", "looking up, dreamy",
    "head tilt, slight smirk", "calm expression, eyes closed",
    "laughing, joyful", "moody red lighting, dark room",
    "neon purple lighting", "soft daylight, window light",
    "graveyard at night background", "old brick wall background",
    "warm candlelight", "black feather angel wings, dramatic",
    "red roses background", "gothic cathedral interior background",
    "close up face portrait, detailed", "upper body, hand near face",
    "full body, standing, fishnet tights", "sitting, relaxed pose",
]

OUT = Path("/content/goth_d_dataset"); OUT.mkdir(exist_ok=True)
imgs = []
for i, var in enumerate(VARIATIONS):
    prompt = BASE.format(trigger=TRIGGER, var=var)
    img = pipe(
        prompt=prompt, negative_prompt=NEG,
        image_embeds=face_emb, image=face_kps,
        controlnet_conditioning_scale=0.8, ip_adapter_scale=0.8,
        num_inference_steps=30, guidance_scale=5.0,
    ).images[0]
    img.save(OUT / f"{i:02d}.png")
    (OUT / f"{i:02d}.txt").write_text(f"{TRIGGER} woman, goth, black dress, {var}")
    imgs.append(img)
    torch.cuda.empty_cache()
    print("generated", i, var)
print("done:", len(imgs), "images")

In [ ]:
# 8. Preview the dataset.
import matplotlib.pyplot as plt

n = len(imgs); cols = 5; rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3 * cols, 3.6 * rows))
for ax, im in zip(axes.ravel(), imgs):
    ax.imshow(im); ax.axis("off")
for ax in axes.ravel()[n:]:
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# 9. Package and download. Keep this zip for Stage 2 (LoRA training).
import shutil
from google.colab import files
shutil.make_archive("/content/goth_d_dataset", "zip", "/content/goth_d_dataset")
files.download("/content/goth_d_dataset.zip")